In [1]:
import pandas as pd
import numpy as np
from collections import Counter
from scipy.stats import entropy, chi2_contingency, spearmanr, pearsonr

In [2]:
def jsd_divergence_between_sets(set1, set2, base=2):

    jsd_results = {}

    dict1 = {list(d.keys())[0]: list(d.values())[0] for d in set1}
    dict2 = {list(d.keys())[0]: list(d.values())[0] for d in set2}

    for question in dict1.keys():
        if question not in dict2:
            continue 

        p_counts = dict1[question]
        q_counts = dict2[question]

        p = np.array([p_counts.get(k, 0) for k in p_counts], dtype=float)
        q = np.array([q_counts.get(k, 0) for k in p_counts], dtype=float)

        p /= p.sum()
        q /= q.sum()

        epsilon = 1e-10
        p = np.clip(p, epsilon, 1)
        q = np.clip(q, epsilon, 1)

        m = 0.5 * (p + q)
        jsd_pq = 0.5 * (entropy(p, m, base=base) + entropy(q, m, base=base)) 
        jsd_results[question] = jsd_pq

    return jsd_results

In [ ]:
AGE_VAR    = "V201507x"
AGE_COL    = "AgeDecade"
AGE_EDGES  = [18, 30, 40, 50, 60, 70, np.inf]
AGE_LABELS = ["18-29", "30-39", "40-49", "50-59", "60-69", "70+"]

def add_age_bins(df):
    """Add the decade-coded column. Values outside 18-80+ (the negative sentinels) become NaN and
    drop out of every group, the same way an unmapped code drops out for any other variable."""
    if AGE_VAR not in df.columns:
        return df
    df = df.copy()
    df[AGE_COL] = pd.cut(pd.to_numeric(df[AGE_VAR], errors="coerce"),
                         bins=AGE_EDGES, labels=False, right=False) + 1
    return df

group_var_dict = {"V201549x" : "Race", 'V202022' : "Discuss Politics", 'V201200' : "Ideology", 'V201231x' : "Party",
                  'V201452' : "Church", 'V201600' : "Gender", 'V202406' : "Political Interest",
                  AGE_COL : "Age"}

groups = {
    'V201549x': {1: 'White', 2: 'Black', 3: 'Asian', 4: 'Native American', 5: 'Hispanic'},
    # 'V201549x': {1: 'White', 2: 'Black', 3: 'Hispanic', 4: 'Asian', 5: 'Native American'},
    'V202022': {
            1: 'Like to discuss politics',
            2: 'Never discuss politics'
            },
    'V201200': {
            1: "Extremely liberal",
            2: "Liberal",
            3: "Slightly liberal",
            4: "Moderate",
            5: "Slightly conservative",
            6: "Conservative",
            7: "Extremely conservative"
        },
    'V201231x': {
            1: "Strong democrat",
            2: "Weak democrat",
            3: "Democrat-leaning indep.",
            4: "Independent",
            5: "Republican-leaning indep.",
            6: "Weak republican",
            7: "Strong republican"
        },
    'V201452': {1: "Attend church", 2: "Does not attend church"},
    'V201600': {1: "Man", 2: "Woman"},
    'V202406': {1: "Very", 2: "Somewhat", 3: "Not very", 4: "Not at all"},
    AGE_COL: dict(enumerate(AGE_LABELS, start=1)),
}

In [4]:
qids = ["V201324","V201416", "V202234", "V202257", "V202287", "V202332", "V202337", "V202348", "V202371", "V202378"]
name_dict = {"V201324" : "Current Economy", "V201416" : "Gay Marriage", "V202234" : "Refugee Allowing",
            "V202257" : "Income Inequality", "V202287" : "Gender Role", "V202332": "Climate Change",
            "V202337": "Gun Regulation", "V202348": "Drug Addiction", "V202371": "Race Diversity", "V202378": "Health Insurance"}

FIVE_POINT_QIDS = {"V201324", "V202332"}          # all other items are 3-point

def informative_codes(qid):
    return set(range(1, 6)) if qid in FIVE_POINT_QIDS else set(range(1, 4))

In [ ]:
_human_cache = {}

def load_human(path):
    """Read the ANES file once and reuse it: it is large and is touched once per grouping variable."""
    if path not in _human_cache:
        _human_cache[path] = add_age_bins(pd.read_csv(path, low_memory=False))
    return _human_cache[path]


def extract_with_nonresponse(source, path, qids, key, missing_as_nonresponse=False):
    info_struct, nr_struct, rates = {}, {}, []
    gvar = group_var_dict[key]
    human_df = load_human(path) if source == "human" else None

    for code_num, group_label in groups[key].items():
        info_struct[group_label], nr_struct[group_label] = [], []
        for qid in qids:
            if source == "llm":
                sub = add_age_bins(pd.read_csv(f"{path}{qid}.csv"))
                sub = sub[sub[key] == code_num]
                resp = pd.to_numeric(sub["Response"], errors="coerce")
            else:
                sub = human_df[human_df[key] == code_num]
                resp = pd.to_numeric(sub[qid], errors="coerce")

            info = informative_codes(qid)
            vc = Counter(resp.dropna())
            informative   = {c: int(vc.get(c, 0)) for c in sorted(info)}
            informative_n = sum(informative.values())
            nonresp_coded = int(sum(cnt for val, cnt in vc.items() if val not in info))
            missing_n     = int(resp.isna().sum())

            nonresp_n  = nonresp_coded + (missing_n if missing_as_nonresponse else 0)
            eligible_n = informative_n + nonresp_n
            rate = nonresp_n / eligible_n if eligible_n > 0 else np.nan

            info_struct[group_label].append({name_dict[qid]: informative})
            nr_struct[group_label].append({name_dict[qid]: {**informative, "NR": nonresp_n}})
            rates.append({"source": source, "grouping_var": gvar, "group": group_label,
                          "question": name_dict[qid], "informative_n": informative_n,
                          "nonresponse_coded": nonresp_coded, "missing_n": missing_n,
                          "nonresponse_n": nonresp_n, "eligible_n": eligible_n,
                          "nonresponse_rate": rate})
    return info_struct, nr_struct, rates


def build_rates(source, path, qids, missing_as_nonresponse=False):
    rows = []
    for key in groups:
        _, _, r = extract_with_nonresponse(source, path, qids, key, missing_as_nonresponse)
        rows += r
    return pd.DataFrame(rows)

In [6]:
def differential_nonresponse(rates_df):
    cols = ["source", "grouping_var", "question", "n_groups_used", "chi2", "p_value",
            "cramers_v", "min_expected", "differential", "note"]
    out = []
    for (src, gvar, q), sub in rates_df.groupby(["source", "grouping_var", "question"]):
        info  = sub["informative_n"].to_numpy(dtype=float)
        nr    = sub["nonresponse_n"].to_numpy(dtype=float)
        table = np.vstack([info, nr])
        keep  = table.sum(axis=0) > 0          # drop groups absent for this question
        table = table[:, keep]
        row = {"source": src, "grouping_var": gvar, "question": q,
               "n_groups_used": int(keep.sum()), "chi2": np.nan, "p_value": np.nan,
               "cramers_v": np.nan, "min_expected": np.nan, "differential": False, "note": ""}
        if table.shape[1] < 2:
            row["note"] = "fewer than 2 non-empty groups"
        elif table[0].sum() == 0:
            row["note"] = "no informative responses"
        elif table[1].sum() == 0:
            row["note"] = "no non-response (cannot be differential)"
        else:
            chi2, p, dof, exp = chi2_contingency(table)
            n = table.sum(); k = min(table.shape) - 1
            row.update(chi2=chi2, p_value=p,
                       cramers_v=(np.sqrt(chi2 / (n * k)) if n > 0 and k > 0 else np.nan),
                       min_expected=exp.min(), differential=bool(p < 0.05))
            if exp.min() < 5:
                row["note"] = "min expected < 5: use Fisher or collapse sparse groups"
        out.append(row)
    df = pd.DataFrame(out, columns=cols)
    if df["p_value"].notna().any():
        df = df.sort_values("p_value", na_position="last")
    return df.reset_index(drop=True)

def jsd_robustness(llm_path, human_path, qids, key):
    """Your JSD (informative only) vs JSD with non-response kept as one bin.
    Same NR definition on both sides so the extra bin is comparable."""
    llm_i, llm_nr, _ = extract_with_nonresponse("llm",   llm_path,   qids, key, False)
    hum_i, hum_nr, _ = extract_with_nonresponse("human", human_path, qids, key, False)
    rows = []
    for g in groups[key].values():
        base = jsd_divergence_between_sets(llm_i[g],  hum_i[g])
        aug  = jsd_divergence_between_sets(llm_nr[g], hum_nr[g])
        for qn in base:
            rows.append({"grouping_var": group_var_dict[key], "group": g, "question": qn,
                         "jsd_informative": base[qn], "jsd_with_nr_bin": aug.get(qn, np.nan),
                         "delta": aug.get(qn, np.nan) - base[qn]})
    return pd.DataFrame(rows)

In [7]:
LLM = "../Publication Results/Gpt4.1-Mini/main_mq_results/full_results_2020_"
HUM = "../data/2020 ANES_test.csv"

# Age needs to be present on both sides, otherwise the Age rows come back empty and silently NaN.
print(f"human: {AGE_VAR} present =", AGE_VAR in load_human(HUM).columns)
print(f"llm:   {AGE_VAR} present =", AGE_VAR in pd.read_csv(f"{LLM}{qids[0]}.csv", nrows=0).columns)

llm_rates   = build_rates("llm",   LLM, qids, missing_as_nonresponse=True)
human_rates = build_rates("human", HUM, qids, missing_as_nonresponse=False)

# The model never abstains, so everything below compares human non-response against a floor of zero.
print("total LLM non-response:", int(llm_rates["nonresponse_n"].sum()))

human: V201507x present = True
llm:   V201507x present = True
total LLM non-response: 0


In [8]:
h = load_human(HUM)

# (A) per-item non-response rate + effective N (ANES non-response = numeric sentinel codes;
#     NaN structural skips are excluded from the denominator, not counted as non-response)
rows = []
for qid in qids:
    v = pd.to_numeric(h[qid], errors="coerce")
    info = list(informative_codes(qid))
    n_sub = int(v.isin(info).sum())
    n_nan = int(v.isna().sum())
    n_nr  = int(len(v) - n_sub - n_nan)
    denom = n_sub + n_nr
    rows.append({"item": name_dict[qid], "N_eff": denom, "nonresponse_%": round(100*n_nr/denom, 1)})
item_tbl = pd.DataFrame(rows).sort_values("nonresponse_%", ascending=False)
print(item_tbl.to_string(index=False))

# (B) available-case vs listwise sample retention (the justification number)
masks = [pd.to_numeric(h[q], errors="coerce").isin(list(informative_codes(q))).values for q in qids]
complete = np.logical_and.reduce(masks)
print(f"total {len(h)}; listwise-complete {complete.sum()} ({100*complete.mean():.1f}%)")

# (C) differential non-response across each conditioning variable, Age included
print(differential_nonresponse(human_rates).to_string(index=False))

             item  N_eff  nonresponse_%
   Drug Addiction   5441           13.7
 Health Insurance   5441           13.7
   Race Diversity   5441           13.4
      Gender Role   5441           13.3
   Gun Regulation   5441           13.2
   Climate Change   5441           13.1
Income Inequality   5441           13.0
 Refugee Allowing   5441           12.8
     Gay Marriage   5441            1.3
  Current Economy   5441            0.3
total 5441; listwise-complete 4557 (83.8%)
source       grouping_var          question  n_groups_used      chi2  p_value  cramers_v  min_expected  differential                                                   note
 human              Party      Gay Marriage              7 33.005029 0.000010   0.078129      5.322545          True                                                       
 human   Discuss Politics   Current Economy              2 11.658817 0.000639   0.049397      1.091042          True min expected < 5: use Fisher or collapse sparse groups
 

In [9]:
# Non-response by age decade, pooled over items. The chi-square in (C) treats the decades as
# unordered, so a rank correlation is what shows whether non-response rises with age.
age_tbl = (human_rates[human_rates["grouping_var"] == "Age"]
           .groupby("group", sort=False)[["nonresponse_n", "eligible_n"]].sum())
age_tbl.insert(0, "respondents", h[AGE_COL].map(groups[AGE_COL]).value_counts().reindex(age_tbl.index))
age_tbl["nonresponse_%"] = (100 * age_tbl["nonresponse_n"] / age_tbl["eligible_n"]).round(1)
print(age_tbl[["respondents", "nonresponse_%"]].to_string())

rho, p = spearmanr(np.arange(len(age_tbl)), age_tbl["nonresponse_%"])
print(f"\ntrend across decades: rho={rho:.3f}, p={p:.4f} (n={len(age_tbl)} decades)")

       respondents  nonresponse_%
group                            
18-29          739           13.1
30-39          910           10.7
40-49          770           10.2
50-59          853           10.7
60-69          999            9.1
70+            914            9.4

trend across decades: rho=-0.841, p=0.0361 (n=6 decades)


In [10]:
df = pd.concat([jsd_robustness(LLM, HUM, qids, k) for k in groups], ignore_index=True)

# 1) how large is the shift when non-response is kept as its own bin
print(df["delta"].describe(percentiles=[.5, .9, .99]).round(4).to_string())

# 2) does the fidelity ordering survive? (the strongest single statement)
print("\nSpearman(informative, with_nr):", round(spearmanr(df.jsd_informative, df.jsd_with_nr_bin).statistic, 4))

# 3) does the headline move at the level actually reported?
agg = df.groupby("grouping_var")[["jsd_informative", "jsd_with_nr_bin"]].mean()
agg["delta"] = agg["jsd_with_nr_bin"] - agg["jsd_informative"]
print("\n" + agg.round(4).sort_values("delta", ascending=False).to_string())

# 4) mechanism: the shift should track the human-minus-model non-response gap
w = (llm_rates.rename(columns={"nonresponse_rate": "llm_nr"})
     .merge(human_rates.rename(columns={"nonresponse_rate": "human_nr"}),
            on=["grouping_var", "group", "question"]))
w["nr_gap"] = (w["human_nr"] - w["llm_nr"]).abs()
merged = df.merge(w, on=["grouping_var", "group", "question"]).dropna(subset=["delta", "nr_gap"])
print("\nPearson(delta, |human_nr - llm_nr|):", round(pearsonr(merged.delta, merged.nr_gap).statistic, 4))

count    350.0000
mean       0.0441
std        0.0302
min       -0.0000
50%        0.0582
90%        0.0760
99%        0.0875
max        0.0913

Spearman(informative, with_nr): 0.9039

                    jsd_informative  jsd_with_nr_bin   delta
grouping_var                                                
Race                         0.0939           0.1515  0.0576
Gender                       0.0912           0.1433  0.0521
Age                          0.0912           0.1430  0.0518
Church                       0.0947           0.1465  0.0518
Party                        0.1189           0.1704  0.0515
Ideology                     0.1001           0.1503  0.0502
Discuss Politics             0.1198           0.1267  0.0070
Political Interest           0.0968           0.0996  0.0028

Pearson(delta, |human_nr - llm_nr|): 0.9936


In [ ]:
d = differential_nonresponse(human_rates)

def summarize(s):
    sig = s[s["differential"]]
    return pd.Series({"items": int(s["differential"].sum()),
                      "max_V": s["cramers_v"].max(),
                      "sparse_sig": int((sig["min_expected"] < 5).sum())})

tbl = (d.groupby("grouping_var")[["differential", "cramers_v", "min_expected"]]
         .apply(summarize)
         .sort_values("items", ascending=False))
tbl["dagger"] = tbl["sparse_sig"] > tbl["items"] / 2   # "mostly from sparse tables"
print(tbl.to_string())


                    items     max_V  sparse_sig  dagger
grouping_var                                           
Party                 9.0  0.078129         0.0   False
Age                   7.0  0.054076         0.0   False
Race                  6.0  0.050779         0.0   False
Discuss Politics      3.0  0.049397         3.0    True
Political Interest    2.0  0.042128         2.0    True
Church                1.0  0.043943         0.0   False
Ideology              1.0  0.055114         1.0    True
Gender                0.0  0.015551         0.0   False
Party & 9/10 & 0.078 \\
Age & 7/10 & 0.054 \\
Race & 6/10 & 0.051 \\
Discuss Politics$^\dagger$ & 3/10 & 0.049 \\
Political Interest$^\dagger$ & 2/10 & 0.042 \\
Church & 1/10 & 0.044 \\
Ideology$^\dagger$ & 1/10 & 0.055 \\
Gender & 0/10 & 0.016 \\
